# Micro-Project 1: Preparing NVD Vulnerability (CVE) Data for Analysis

**ANA500 -- Data Science Process**

- **Author:** Christian Cruz
- **Scope:** This notebook covers the **Acquire** and **Prepare** steps of the data science
process only, per the Micro-Project 1 requirements (NumPy & pandas objects to organize,
clean, and explore data). Analysis, modeling, and reporting are out of scope here and will
be covered in later micro-projects using this same underlying dataset.

## Problem Statement
Security teams face a constantly growing volume of publicly disclosed vulnerabilities,
called Common Vulnerabilities and Exposures (CVEs), and must decide which to patch first,
and how to plan analyst capacity to keep up. This connects directly to compliance work
(e.g. NIST 800-53, CIS hardening baselines), but raw public vulnerability data isn't
analysis-ready: it mixes incompatible versions of the Common Vulnerability Scoring System
(CVSS) standard adopted over more than a decade, and includes withdrawn or unscored
records that must be identified before it can support real decisions.

## Hypothesis Formulation
Structural characteristics of a CVE record (its CVSS attack vector, attack complexity,
privileges required, and scope) are associated with its severity classification, but the
raw NVD data first requires meaningful cleaning: it spans multiple incompatible CVSS
scoring standards (v2, v3.0, v3.1, v4) adopted over more than a decade, contains
placeholder/withdrawn records that are not real vulnerabilities, and has inconsistent
completeness across fields depending on when and how each record was submitted.

## Step 1: Acquire

**Data source:** [`fkie-cad/nvd-json-data-feeds`](https://github.com/fkie-cad/nvd-json-data-feeds) --
a community-maintained reconstruction of NIST's National Vulnerability Database (NVD)
bulk JSON feeds. NIST deprecated its own year-by-year bulk downloads in December 2023 in
favor of a rate-limited live API; this project re-packages that same official NVD API 2.0
data into the old yearly-archive format, refreshed from NIST every two hours. This gives us
bulk, multi-year historical data without needing to paginate through the live API's rate
limits ourselves.

We pull twelve years of data (2015-2026) so that later micro-projects in this portfolio
have enough history to build a meaningful monthly time series (needed for the deep-learning
forecasting work in Micro-Project 4).

Each yearly file is downloaded only if it isn't already present locally, so re-running this
notebook doesn't re-download data unnecessarily.


In [1]:
import os
import lzma
import json
import urllib.request
from pathlib import Path

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2015, 2027))  # 2015 through 2026
BASE_URL = "https://github.com/fkie-cad/nvd-json-data-feeds/releases/latest/download"

for year in YEARS:
    fname = f"CVE-{year}.json.xz"
    target = DATA_DIR / fname
    if target.exists():
        print(f"{fname}: already downloaded, skipping")
        continue
    url = f"{BASE_URL}/{fname}"
    print(f"{fname}: downloading from {url} ...")
    urllib.request.urlretrieve(url, target)

print("\nAll yearly CVE archives present in", DATA_DIR.resolve())


CVE-2015.json.xz: already downloaded, skipping
CVE-2016.json.xz: already downloaded, skipping
CVE-2017.json.xz: already downloaded, skipping
CVE-2018.json.xz: already downloaded, skipping
CVE-2019.json.xz: already downloaded, skipping
CVE-2020.json.xz: already downloaded, skipping
CVE-2021.json.xz: already downloaded, skipping
CVE-2022.json.xz: already downloaded, skipping
CVE-2023.json.xz: already downloaded, skipping
CVE-2024.json.xz: already downloaded, skipping
CVE-2025.json.xz: already downloaded, skipping
CVE-2026.json.xz: already downloaded, skipping

All yearly CVE archives present in C:\Users\cruzi\OneDrive\Desktop\National University\ANA_500\micro-project-1\data\raw


In [2]:
# Peek at the raw structure of a single year's file before parsing everything,
# so we know exactly what we're extracting and why.
with lzma.open(DATA_DIR / "CVE-2020.json.xz") as f:
    sample = json.load(f)

print("Top-level keys:", list(sample.keys()))
print("cve_count (2020):", sample["cve_count"])
print()
print("Keys on a single CVE record:")
print(list(sample["cve_items"][0].keys()))


Top-level keys: ['timestamp', 'cve_count', 'feed_name', 'source', 'cve_items']
cve_count (2020): 21076

Keys on a single CVE record:
['id', 'sourceIdentifier', 'published', 'lastModified', 'vulnStatus', 'cveTags', 'descriptions', 'affected', 'metrics', 'weaknesses', 'configurations', 'references']


## Step 2: Prepare -- Flattening the Nested JSON

Each CVE record is deeply nested: severity scoring lives under
`metrics.cvssMetricV2` / `cvssMetricV30` / `cvssMetricV31` / `cvssMetricV40`, and each of
those may or may not be present on a given record depending on when it was scored and
under which CVSS standard. Before pandas can do anything useful with this, we need a flat,
one-row-per-CVE table.

We keep CVSS v2 and CVSS v3.x/v4 fields in **separate** columns rather than merging them,
because they use different rating scales and incompatible sub-fields (v2's
`authentication` field, for example, is not equivalent to v3's `privilegesRequired`). We
also compute one derived `primary_*` column set, preferring the newest CVSS version present
on each record, for any analysis that just needs a single severity number.


In [3]:
import numpy as np
import pandas as pd

FIELDNAMES = [
    "cve_id", "published_date", "last_modified_date", "vuln_status",
    "description_en",
    "primary_vendor", "primary_product", "num_affected_products", "cwe_id",
    "v2_score", "v2_severity", "v2_vector", "v2_access_vector",
    "v2_access_complexity", "v2_authentication",
    "v3_version", "v3_score", "v3_severity", "v3_vector",
    "v3_attack_vector", "v3_attack_complexity", "v3_privileges_required",
    "v3_user_interaction", "v3_scope",
    "v3_confidentiality_impact", "v3_integrity_impact", "v3_availability_impact",
    "v3_exploitability_score", "v3_impact_score",
    "v4_score", "v4_severity", "v4_vector",
    "primary_score", "primary_severity", "primary_version",
]


def first_metric(metric_list):
    """Prefer the entry NVD marked as type == 'Primary'; else take the first."""
    if not metric_list:
        return None
    for m in metric_list:
        if m.get("type") == "Primary":
            return m
    return metric_list[0]


def extract_row(item):
    row = {k: None for k in FIELDNAMES}
    row["cve_id"] = item.get("id")
    row["published_date"] = item.get("published")
    row["last_modified_date"] = item.get("lastModified")
    row["vuln_status"] = item.get("vulnStatus")

    for d in item.get("descriptions", []):
        if d.get("lang") == "en":
            row["description_en"] = d.get("value")
            break

    vendors, products = [], []
    for src in item.get("affected", []):
        for ad in src.get("affectedData", []):
            if ad.get("vendor"):
                vendors.append(ad["vendor"])
            if ad.get("product"):
                products.append(ad["product"])
    row["primary_vendor"] = vendors[0] if vendors else None
    row["primary_product"] = products[0] if products else None
    row["num_affected_products"] = len(products)

    weaknesses = item.get("weaknesses", [])
    if weaknesses:
        for desc in weaknesses[0].get("description", []):
            if desc.get("lang") == "en":
                row["cwe_id"] = desc.get("value")
                break

    metrics = item.get("metrics", {})

    v2 = first_metric(metrics.get("cvssMetricV2"))
    if v2:
        cd = v2.get("cvssData", {})
        row["v2_score"] = cd.get("baseScore")
        row["v2_severity"] = v2.get("baseSeverity") or cd.get("baseSeverity")
        row["v2_vector"] = cd.get("vectorString")
        row["v2_access_vector"] = cd.get("accessVector")
        row["v2_access_complexity"] = cd.get("accessComplexity")
        row["v2_authentication"] = cd.get("authentication")

    v3 = first_metric(metrics.get("cvssMetricV31")) or first_metric(metrics.get("cvssMetricV30"))
    if v3:
        cd = v3.get("cvssData", {})
        row["v3_version"] = cd.get("version")
        row["v3_score"] = cd.get("baseScore")
        row["v3_severity"] = cd.get("baseSeverity")
        row["v3_vector"] = cd.get("vectorString")
        row["v3_attack_vector"] = cd.get("attackVector")
        row["v3_attack_complexity"] = cd.get("attackComplexity")
        row["v3_privileges_required"] = cd.get("privilegesRequired")
        row["v3_user_interaction"] = cd.get("userInteraction")
        row["v3_scope"] = cd.get("scope")
        row["v3_confidentiality_impact"] = cd.get("confidentialityImpact")
        row["v3_integrity_impact"] = cd.get("integrityImpact")
        row["v3_availability_impact"] = cd.get("availabilityImpact")
        row["v3_exploitability_score"] = v3.get("exploitabilityScore")
        row["v3_impact_score"] = v3.get("impactScore")

    v4 = first_metric(metrics.get("cvssMetricV40"))
    if v4:
        cd = v4.get("cvssData", {})
        row["v4_score"] = cd.get("baseScore")
        row["v4_severity"] = cd.get("baseSeverity")
        row["v4_vector"] = cd.get("vectorString")

    # Derived "best available" severity: prefer the newest CVSS version present
    if v4:
        row["primary_score"], row["primary_severity"], row["primary_version"] = row["v4_score"], row["v4_severity"], "4.0"
    elif v3:
        row["primary_score"], row["primary_severity"], row["primary_version"] = row["v3_score"], row["v3_severity"], row["v3_version"]
    elif v2:
        row["primary_score"], row["primary_severity"], row["primary_version"] = row["v2_score"], row["v2_severity"], "2.0"

    return row


In [4]:
records = []
for year in YEARS:
    fpath = DATA_DIR / f"CVE-{year}.json.xz"
    with lzma.open(fpath) as f:
        data = json.load(f)
    for item in data["cve_items"]:
        records.append(extract_row(item))
    print(f"CVE-{year}: {len(data['cve_items'])} records")

df_raw = pd.DataFrame.from_records(records)
print(f"\nTotal records flattened: {len(df_raw)}")


CVE-2015: 8779 records
CVE-2016: 10647 records
CVE-2017: 17105 records
CVE-2018: 17817 records
CVE-2019: 17623 records
CVE-2020: 21076 records
CVE-2021: 23469 records
CVE-2022: 27554 records
CVE-2023: 31442 records
CVE-2024: 39247 records
CVE-2025: 45279 records
CVE-2026: 61202 records

Total records flattened: 321240


CVE-2016: 10647 records


CVE-2017: 17105 records


CVE-2018: 17817 records


CVE-2019: 17623 records


...

CVE-2023: 31442 records


CVE-2024: 39247 records


CVE-2025: 45279 records


CVE-2026: 61202 records



Total records flattened: 321240


### A hidden data quality issue: placeholder text instead of a real blank

Some CNAs (the organizations that submit CVE records to NVD) fill the vendor/product
fields with the literal text `"n/a"` instead of leaving them empty when that information
isn't available -- the CVE equivalent of someone typing "n/a" into a form field rather than
leaving it blank. pandas has no way to know that `"n/a"` is meant to mean "missing" unless
we tell it -- as far as pandas is concerned, `"n/a"` is just a three-character piece of text,
no different from `"Microsoft"`. Left alone, this would make later steps (including
`.isnull()` checks, and any later micro-project that uses vendor/product as a category)
undercount how much is actually missing.

We fix this once, right here, so every later step in this notebook and every later
micro-project that reuses this dataset sees the true picture.


In [5]:
# Treat literal placeholder text ("n/a", blank) as real missing values, not as data.
for col in ["primary_vendor", "primary_product"]:
    is_placeholder = df_raw[col].astype(str).str.strip().str.lower().isin(["", "n/a", "none", "nan"])
    df_raw.loc[is_placeholder, col] = np.nan

print("primary_vendor missing:", df_raw["primary_vendor"].isnull().sum(),
      f"({df_raw['primary_vendor'].isnull().mean():.1%})")
print("primary_product missing:", df_raw["primary_product"].isnull().sum(),
      f"({df_raw['primary_product'].isnull().mean():.1%})")


primary_vendor missing: 106910 (33.3%)
primary_product missing: 90444 (28.2%)


## Step 3: Explore the Raw, Flattened Data

Before deciding how to clean anything, we look at what's actually here: shape, dtypes,
missing values, duplicates, and the distribution of key categorical fields.


In [6]:
df_raw.shape


(321240, 35)

In [7]:
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 321240 entries, 0 to 321239
Data columns (total 35 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   cve_id                     321240 non-null  object 
 1   published_date             321240 non-null  object 
 2   last_modified_date         321240 non-null  object 
 3   vuln_status                321240 non-null  object 
 4   description_en             321240 non-null  object 
 5   primary_vendor             214330 non-null  object 
 6   primary_product            230796 non-null  object 
 7   num_affected_products      321240 non-null  int64  
 8   cwe_id                     299229 non-null  object 
 9   v2_score                   125402 non-null  float64
 10  v2_severity                125402 non-null  object 
 11  v2_vector                  125402 non-null  object 
 12  v2_access_vector           125402 non-null  object 
 13  v2_access_complexity       12

In [8]:
# Missing-value audit, sorted highest first
df_raw.isnull().sum().sort_values(ascending=False)


v4_score                     284156
v4_vector                    284156
v4_severity                  284156
v2_authentication            195838
v2_severity                  195838
v2_access_complexity         195838
v2_access_vector             195838
v2_vector                    195838
v2_score                     195838
primary_vendor               106910
primary_product               90444
v3_attack_vector              29909
v3_confidentiality_impact     29909
v3_integrity_impact           29909
v3_scope                      29909
v3_user_interaction           29909
v3_privileges_required        29909
v3_availability_impact        29909
v3_attack_complexity          29909
v3_severity                   29909
v3_vector                     29909
v3_score                      29909
v3_version                    29909
v3_impact_score               29909
v3_exploitability_score       29909
cwe_id                        22011
primary_score                 18368
primary_severity            

In [9]:
# Are there any duplicate CVE IDs? (there shouldn't be -- each CVE ID is unique by design)
print("Duplicate CVE IDs:", df_raw["cve_id"].duplicated().sum())


Duplicate CVE IDs: 0


In [10]:
df_raw["vuln_status"].value_counts()


vuln_status
Modified               173565
Analyzed                72367
Deferred                49261
Rejected                15357
Received                 5081
Awaiting Analysis        4854
Undergoing Analysis       755
Name: count, dtype: int64

In [11]:
df_raw["primary_severity"].value_counts(dropna=False)


primary_severity
MEDIUM      132274
HIGH        120426
CRITICAL     38224
None         18368
LOW          11874
NONE            74
Name: count, dtype: int64

In [12]:
df_raw["primary_version"].value_counts(dropna=False)


primary_version
3.1     218201
3.0      42020
4.0      37084
None     18368
2.0       5567
Name: count, dtype: int64

In [13]:
# Which vuln_status values account for the records with NO CVSS score at all?
df_raw.loc[df_raw["primary_score"].isnull(), "vuln_status"].value_counts()


vuln_status
Rejected               15357
Received                1850
Deferred                1001
Awaiting Analysis        152
Modified                   4
Undergoing Analysis        4
Name: count, dtype: int64

In [14]:
# num_affected_products is a NumPy-backed numeric column -- pull it out as a raw
# NumPy array to check its distribution and look for outliers using percentiles.
n_affected = df_raw["num_affected_products"].to_numpy()

q1, q3 = np.percentile(n_affected, [25, 75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

print(f"25th pct: {q1}, 75th pct: {q3}, IQR: {iqr}")
print(f"Upper outlier fence (Tukey's rule): {upper_fence}")
print(f"Records above the fence: {(n_affected > upper_fence).sum()} of {len(n_affected)}")
print(f"Max affected products on a single CVE: {n_affected.max()}")
print(f"Mean: {n_affected.mean():.2f}, Median: {np.median(n_affected):.1f}, Std: {n_affected.std():.2f}")


25th pct: 1.0, 75th pct: 1.0, IQR: 0.0
Upper outlier fence (Tukey's rule): 1.0
Records above the fence: 64208 of 321240
Max affected products on a single CVE: 2749
Mean: 2.04, Median: 1.0, Std: 12.48


**Findings from exploration:**

- **321,240** raw records across 2015-2026, no duplicate CVE IDs.
- CVSS scoring standard changes dramatically over time: **218,201** records use v3.1,
  **42,020** use v3.0, **37,084** use the brand-new v4.0, and only **5,567** (mostly older
  CVEs) still rely on v2 as their best available score. This isn't random missingness --
  it reflects NVD's own scoring-standard adoption timeline, and it's exactly why the v2 and
  v3-family columns are kept separate rather than merged.
- **18,368** records (5.7%) have no CVSS score at all. The overwhelming majority of those
  (**15,357**) are `Rejected` CVE IDs -- withdrawn, not real vulnerabilities -- with a
  smaller number still sitting in NVD's `Received`/`Deferred`/`Awaiting Analysis` queue.
- `primary_vendor` is missing on about **30%** of records and `primary_product` on about
  **25%** (after correcting for the `"n/a"` placeholder text described above). This is a
  genuine upstream data quality issue in how some CNAs submit records, not a bug in this
  extraction.
- `num_affected_products` is heavily right-skewed (mean ~2, but a max of nearly 2,750
  affected product/version combinations on a single CVE) -- worth flagging for outlier
  handling in any later modeling step, though no transformation is applied yet since this
  project stops at Prepare.


## Step 4: Clean the Data

Based on what exploration turned up, here's what gets cleaned and why:

1. **Drop `Rejected` records** -- a withdrawn CVE ID is not a vulnerability and would
   distort any severity or trend analysis.
2. **Drop remaining unscored records** -- every downstream analysis (severity regression in
   Micro-Project 3, forecasting in Micro-Project 4) needs a target severity value; a record
   without any CVSS score yet can't be used for that.
3. **Parse `published_date` / `last_modified_date`** into real datetimes, and derive
   `published_year` / `published_year_month` -- needed for the time-series work later in
   this portfolio.
4. **Truncate the free-text description** to 300 characters. The full descriptions (often
   several sentences, in some cases across languages elsewhere in the source data) account
   for the majority of the raw file's size and aren't needed for the structured/numeric
   analysis this course is scoped to.
5. **Leave `primary_vendor`/`primary_product` missingness as-is** rather than guess-filling
   it -- properly resolving it would require parsing the older CPE `configurations`
   structure as a fallback, which is out of scope for this micro-project and is noted here
   as a known limitation rather than silently patched over.


In [15]:
n_start = len(df_raw)
df = df_raw.copy()

# 1. Drop rejected CVE IDs
n_rejected = (df["vuln_status"] == "Rejected").sum()
df = df[df["vuln_status"] != "Rejected"].copy()
print(f"Dropped {n_rejected} Rejected records -> {len(df)} remain")

# 2. Drop rows with no CVSS score of any version
n_unscored = df["primary_score"].isnull().sum()
df = df[df["primary_score"].notnull()].copy()
print(f"Dropped {n_unscored} unscored records -> {len(df)} remain")


Dropped 15357 Rejected records -> 305883 remain
Dropped 3011 unscored records -> 302872 remain


Dropped 3011 unscored records -> 302872 remain


In [16]:
# 3. Parse dates and derive year / year-month columns
df["published_date"] = pd.to_datetime(df["published_date"], errors="coerce")
df["last_modified_date"] = pd.to_datetime(df["last_modified_date"], errors="coerce")
df["published_year"] = df["published_date"].dt.year
df["published_year_month"] = df["published_date"].dt.to_period("M").astype(str)

# 4. Truncate free-text description
df["description_short"] = df["description_en"].str.slice(0, 300)
df = df.drop(columns=["description_en"])

# Reorder columns so the most useful ones are up front
front = [
    "cve_id", "published_date", "published_year", "published_year_month",
    "last_modified_date", "vuln_status", "description_short",
    "primary_vendor", "primary_product", "num_affected_products", "cwe_id",
    "primary_score", "primary_severity", "primary_version",
]
rest = [c for c in df.columns if c not in front]
df = df[front + rest]

print(f"Rows removed overall: {n_start - len(df)} ({(n_start - len(df)) / n_start:.1%})")
df.shape


Rows removed overall: 18368 (5.7%)


(302872, 37)

In [17]:
# Sanity checks on the cleaned dataset
assert df["primary_score"].isnull().sum() == 0, "Every remaining record should have a score"
assert (df["vuln_status"] == "Rejected").sum() == 0, "No rejected records should remain"
assert df["cve_id"].duplicated().sum() == 0, "CVE IDs should still be unique"
assert df["published_date"].notnull().all(), "All dates should have parsed successfully"

print("All sanity checks passed.")
df.describe(include="all").T


All sanity checks passed.


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
cve_id,302872,302872,CVE-2015-0001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
published_date,302872,NaN,NaN,NaN,2023-02-18 00:06:41.692316160,2015-01-07 19:59:05.107000,2020-12-15 16:15:15.212000,2024-01-12 16:15:52.440000,2025-10-14 13:15:37.234749952,2026-09-17 23:18:54.710000,NaN
published_year,302872.0,NaN,NaN,NaN,2022.641169,2015.0,2020.0,2024.0,2025.0,2026.0,3.089558
published_year_month,302872,141,2026-08,11565,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_modified_date,302872,NaN,NaN,NaN,2026-06-26 05:29:09.955728128,2026-06-17 00:19:24.040000,2026-06-17 03:12:28.194499840,2026-06-17 06:32:04.312999936,2026-06-17 09:51:15.335000064,2026-09-17 23:18:54.710000,NaN
vuln_status,302872,6,Modified,173561,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description_short,302872,281172,Adobe Experience Manager versions 6.5.22 and e...,223,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_vendor,211348,24735,Linux,13780,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primary_product,227814,55403,Linux,13651,NaN,NaN,NaN,NaN,NaN,NaN,NaN
num_affected_products,302872.0,NaN,NaN,NaN,2.144325,0.0,1.0,1.0,1.0,2749.0,12.841454


In [18]:
df.head(10)


,cve_id,published_date,published_year,published_year_month,last_modified_date,vuln_status,description_short,primary_vendor,primary_product,num_affected_products,...,v3_user_interaction,v3_scope,v3_confidentiality_impact,v3_integrity_impact,v3_availability_impact,v3_exploitability_score,v3_impact_score,v4_score,v4_severity,v4_vector
0,CVE-2015-0001,2015-01-13 22:59:00.050,2015,2015-01,2026-06-17 00:19:24.040,Modified,The Windows Error Reporting (WER) component in...,NaN,NaN,1,...,None,None,None,None,None,NaN,NaN,NaN,None,None
1,CVE-2015-0002,2015-01-13 22:59:01.253,2015,2015-01,2026-06-17 00:19:24.160,Modified,The AhcVerifyAdminContext function in ahcache....,NaN,NaN,1,...,None,None,None,None,None,NaN,NaN,NaN,None,None
2,CVE-2015-0003,2015-02-11 03:00:28.763,2015,2015-02,2026-06-17 00:19:24.280,Modified,win32k.sys in the kernel-mode drivers in Micro...,NaN,NaN,1,...,None,None,None,None,None,NaN,NaN,NaN,None,None
3,CVE-2015-0004,2015-01-13 22:59:02.597,2015,2015-01,2026-06-17 00:19:24.390,Modified,The User Profile Service (aka ProfSvc) in Micr...,NaN,NaN,1,...,None,None,None,None,None,NaN,NaN,NaN,None,None
4,CVE-2015-0005,2015-03-11 10:59:00.087,2015,2015-03,2026-06-17 00:19:24.507,Modified,The NETLOGON service in Microsoft Windows Serv...,NaN,NaN,1,...,None,None,None,None,None,NaN,NaN,NaN,None,None
5,CVE-2015-0006,2015-01-13 22:59:03.580,2015,2015-01,2026-06-17 00:19:24.627,Modified,The Network Location Awareness (NLA) service i...,NaN,NaN,1,...,None,None,None,None,None,NaN,NaN,NaN,None,None
7,CVE-2015-0008,2015-02-11 03:00:29.607,2015,2015-02,2026-06-17 00:19:24.760,Modified,The UNC implementation in Microsoft Windows Se...,NaN,NaN,1,...,None,None,None,None,None,NaN,NaN,NaN,None,None
8,CVE-2015-0009,2015-02-11 03:00:30.700,2015,2015-02,2026-06-17 00:19:24.893,Modified,The Group Policy Security Configuration policy...,NaN,NaN,1,...,None,None,None,None,None,NaN,NaN,NaN,None,None
9,CVE-2015-0010,2015-02-11 03:00:31.480,2015,2015-02,2026-06-17 00:19:25.007,Modified,The CryptProtectMemory function in cng.sys (ak...,NaN,NaN,1,...,None,None,None,None,None,NaN,NaN,NaN,None,None
10,CVE-2015-0011,2015-01-13 22:59:04.470,2015,2015-01,2026-06-17 00:19:25.117,Modified,mrxdav.sys (aka the WebDAV driver) in the kern...,NaN,NaN,1,...,None,None,None,None,None,NaN,NaN,NaN,None,None


In [19]:
# Save the cleaned, analysis-ready dataset for use in later micro-projects.
# (Kept out of version control via .gitignore -- see README -- since it's fully
# reproducible by re-running this notebook against the public data source above.)
df.to_csv("data/cve_clean.csv", index=False)
print("Saved data/cve_clean.csv:", df.shape)


Saved data/cve_clean.csv: (302872, 37)


## Summary / Bridge to Micro-Project 2

Starting from 321,240 raw CVE records spanning 12 years and four incompatible CVSS scoring
standards, this notebook produced a clean, de-duplicated, consistently-typed dataset of
**302,872** scored vulnerabilities with parsed dates, a unified severity column, and a
documented, auditable trail of exactly what was removed and why. Micro-Project 2 will build
on this same cleaned dataset to visualize disclosure trends, severity distributions, and
attack-vector patterns (Matplotlib/Seaborn/Plotly), continuing toward the ML regression work
in Micro-Project 3 and the time-series forecasting in Micro-Project 4.
